# Plot of M1 and S1 PSD

Plot of M1 and S1 PSD in TVMB-only and TVMB-NEST co-simulations, as:
- mean and individual lines, 
- mean and STD 
- average content in gamma and theta bands.

**Authors:** Alice Geminiani ([alice.geminiani@unipv.it](mailto:alice.geminiani@unipv.it)) and GitHub Copilot with Claude Opus 4.6

In [ ]:
import pickle
import numpy as np

# Load the REST FIT pickle file
file_path = r"c:\Users\alice\Documents\TVB_NEST_cerebellum\FinalResultsDP\PSD plot\PSDsMeanSimsFIT_REST.pkl"
with open(file_path, 'rb') as f:
    data_rest = pickle.load(f)

# Load the COSIM pickle file
file_cosim = r"c:\Users\alice\Documents\TVB_NEST_cerebellum\FinalResultsDP\PSD plot\PSD_Final10reps_COSIM_noise1e4.pkl"
with open(file_cosim, 'rb') as f:
    data_cosim_raw = pickle.load(f)

# Convert xarray-like COSIM data to numpy array
# Structure: dims = ('G', 'Condition', 'Repetition', 'Region', 'Frequency')
# G values: [4, 5, 6, 7, 8]
# Conditions: ['CEREBON', 'CEREBOFF']
# Repetitions: 10
# Regions: 20
# Frequencies: 96 bins (5-100 Hz)

# Extract coordinates
G_values = data_cosim_raw['coords']['G']['data']  # [4, 5, 6, 7, 8]
conditions = data_cosim_raw['coords']['Condition']['data']  # ['CEREBON', 'CEREBOFF']
regions = data_cosim_raw['coords']['Region']['data']  # 20 region names
freqs_cosim = np.array(data_cosim_raw['coords']['Frequency']['data'])  # 5-100 Hz

# Convert nested list to numpy array
data_array = np.array(data_cosim_raw['data'])
print(f"COSIM data array shape: {data_array.shape}")
print(f"Expected dims: (G={len(G_values)}, Condition={len(conditions)}, Repetition=10, Region={len(regions)}, Frequency={len(freqs_cosim)})")

# Create data_cosim dict in a format compatible with the plotting code
# We need: data_cosim[iG] = (10, 172) array - but COSIM has different structure
# For compatibility, we'll extract G=6 for 'CEREBON' condition and select M1/S1 regions

# Region mapping for M1 and S1:
# M1: 'Right Primary motor area' (idx 0), 'Left Primary motor area' (idx 1)
# S1: 'Right Primary somatosensory area, barrel field' (idx 18), 'Left Primary somatosensory area, barrel field' (idx 19)
region_idx_map = {
    'RM1': 0,  # Right Primary motor area
    'LM1': 1,  # Left Primary motor area
    'RS1': 18, # Right Primary somatosensory area, barrel field
    'LS1': 19  # Left Primary somatosensory area, barrel field
}

print(f"\nRegion indices being used:")
for name, idx in region_idx_map.items():
    print(f"  {name}: idx {idx} = '{regions[idx]}'")

# Store the raw COSIM data array and metadata for plotting
data_cosim = {
    'data_array': data_array,
    'G_values': G_values,
    'conditions': conditions,
    'regions': regions,
    'freqs': freqs_cosim,
    'region_idx_map': region_idx_map
}

# Inspect the REST data
print("\nREST FIT Data:")
print(f"Type: {type(data_rest)}")
print(f"Keys: {list(data_rest.keys())[:5]}... (total {len(data_rest)} keys)")
if 'PSD_target' in data_rest:
    print(f"PSD_target shape: {data_rest['PSD_target'].shape}")
if '6' in data_rest:
    print(f"iG='6' shape: {data_rest['6'].shape}")

print("\nCOSIM Data (processed):")
print(f"Data array shape: {data_cosim['data_array'].shape}")
print(f"G values: {data_cosim['G_values']}")
print(f"Conditions: {data_cosim['conditions']}")
print(f"Frequency range: {data_cosim['freqs'][0]} - {data_cosim['freqs'][-1]} Hz")

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import matplotlib.pyplot as plt
import numpy as np
from matplotlib import cm
from matplotlib.transforms import blended_transform_factory
from matplotlib.lines import Line2D
from NESTlesions.plot_utils import save_figure_multi_format

# ── Color constants ──────────────────────────────────────────────────────────
viridis = cm.get_cmap('viridis')
color_rest = viridis(0.1)        # Purple for TVMB-only simulations
color_cosim = viridis(0.75)      # Green for Co-sim simulations
color_exp = viridis(0.35)        # Blue for experimental
alpha_rest = 0.4                 # Alpha for TVMB-only lines

# ── Target G and condition ───────────────────────────────────────────────────
target_G = 6
target_condition = 'CEREBON'

# ── Extract REST data ────────────────────────────────────────────────────────
psd_target = data_rest['PSD_target']
total_freq_bins = psd_target.shape[0]
Nf = int(total_freq_bins / 4)  # 43 frequency bins per region
psd_data_rest = data_rest[target_G]
print(f"REST PSD data shape for G={target_G}: {psd_data_rest.shape}")

# ── Extract COSIM data ───────────────────────────────────────────────────────
G_idx = data_cosim['G_values'].index(target_G)
cond_idx = data_cosim['conditions'].index(target_condition)
print(f"COSIM: Using G={target_G} (idx {G_idx}), Condition='{target_condition}' (idx {cond_idx})")
cosim_data_full = data_cosim['data_array'][G_idx, cond_idx, :, :, :]
print(f"COSIM extracted data shape: {cosim_data_full.shape}")

# ── Frequency vectors ────────────────────────────────────────────────────────
freqs_rest = np.arange(5, 48, 1)
freqs_cosim = data_cosim['freqs']
freq_mask = freqs_cosim <= 47
freqs = freqs_rest
region_idx_cosim = data_cosim['region_idx_map']

# ── Plot configurations ─────────────────────────────────────────────────────
# (region_type, dataset_name, rest_indices, cosim_names)
plot_configs = [
    ('M1', 'REST FIT', [0, 1], ['RM1', 'LM1']),
    ('S1', 'REST FIT', [2, 3], ['RS1', 'LS1']),
    ('M1', 'COSIM',    [0, 1], ['RM1', 'LM1']),
    ('S1', 'COSIM',    [2, 3], ['RS1', 'LS1']),
]

# ── Figure style (reference: first two plots) ───────────────────────────────
plt.rcParams.update({
    'font.size': 14,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'font.family': 'sans-serif'
})


# ═════════════════════════════════════════════════════════════════════════════
#  Helper functions
# ═════════════════════════════════════════════════════════════════════════════

def normalize_psd(psd_data):
    """Normalize PSD by dividing by total power (sum of PSD values)."""
    return psd_data / np.sum(psd_data)


def collect_rest_data(rest_indices, normalize=False):
    """Collect REST PSD data for all repetitions and hemispheres.

    Returns
    -------
    np.ndarray of shape (n_reps * n_hemispheres, Nf)
    """
    all_data = []
    for rep in range(psd_data_rest.shape[0]):
        for region_idx in rest_indices:
            start_idx = region_idx * Nf
            end_idx = start_idx + Nf
            rep_data = psd_data_rest[rep, start_idx:end_idx]
            if normalize:
                rep_data = normalize_psd(rep_data)
            all_data.append(rep_data)
    return np.array(all_data)


def collect_cosim_data(cosim_names, normalize=False):
    """Collect COSIM PSD data for all repetitions and hemispheres.

    Returns
    -------
    np.ndarray of shape (n_reps * n_hemispheres, n_freqs)
    """
    all_data = []
    for rep in range(cosim_data_full.shape[0]):
        for region_name in cosim_names:
            region_idx = region_idx_cosim[region_name]
            rep_data = cosim_data_full[rep, region_idx, freq_mask]
            if normalize:
                rep_data = normalize_psd(rep_data)
            all_data.append(rep_data)
    return np.array(all_data)


def get_experimental_target(rest_indices, normalize=False):
    """Compute mean experimental target PSD across hemispheres."""
    target_data = []
    for region_idx in rest_indices:
        start_idx = region_idx * Nf
        end_idx = start_idx + Nf
        target_data.append(psd_target[start_idx:end_idx])
    target_mean = np.mean(target_data, axis=0)
    if normalize:
        target_mean = normalize_psd(target_mean)
    return target_mean


def style_psd_axis(ax, plot_idx, region_type, dataset_name,
                   ylabel='PSD', xlim=(0, 50), ylim=None,
                   show_legend=False, grid_which='both'):
    """Apply consistent styling to a PSD axis (matching reference figures).

    Includes: axis labels, frequency-band rectangles with Greek letters,
    vertical band-boundary lines, spine removal, grid, title, and
    optional legend on the last panel.
    """
    ax.set_xlabel('Frequency (Hz)')
    if plot_idx == 0:
        ax.set_ylabel(ylabel)
    else:
        ax.set_yticklabels([])
    ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)

    # Vertical lines at frequency-band boundaries
    for x in [6, 12, 25]:
        ax.axvline(x=x, color='black', linestyle='--', alpha=0.5, linewidth=0.8)

    # Frequency-band rectangles and Greek-letter labels
    trans = blended_transform_factory(ax.transData, ax.transAxes)
    band_height = 0.03
    theta_y = 0.0
    beta_y = theta_y + band_height
    bands = [
        (6,  12, 'θ', theta_y),
        (12, 25, 'β', beta_y),
        (25, 47, 'γ', theta_y),
    ]
    for start, end, label, y_pos in bands:
        rect = plt.Rectangle((start, y_pos), end - start, band_height,
                              transform=trans, facecolor='darkgray',
                              edgecolor='none', alpha=0.5, clip_on=False)
        ax.add_patch(rect)
        label_y = y_pos + band_height + 0.02
        ax.text((start + end) / 2, label_y, label, transform=trans,
                ha='center', va='bottom', fontsize=14)

    ax.set_title(f'{region_type} - {dataset_name}')

    if show_legend:
        legend_elements = [
            Line2D([0], [0], color=color_rest, linewidth=1.5, label='TVMB-only'),
            Line2D([0], [0], color=color_cosim, linewidth=1.5, label='Co-sim'),
            Line2D([0], [0], color=color_exp, linewidth=2, label='Exp'),
        ]
        ax.legend(handles=legend_elements, loc='upper right', fontsize=10)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.3, which=grid_which)


# ── Frequency-band masks (on the shared `freqs` vector) ─────────────────────
theta_mask = (freqs >= 6) & (freqs <= 12)
gamma_mask = (freqs >= 25) & (freqs <= 47)

print("Helper functions defined.")

In [ ]:
# Individual lines – NORMALIZED – separate 1×3 per dataset
# Row definitions: (dataset, color, alpha, sim_label)
row_defs = [
    ('REST FIT', color_rest, alpha_rest, 'TVMB-only'),
    ('COSIM',    color_cosim, 0.7,       'Co-sim'),
]

# Region definitions: (col, region_type, rest_indices, cosim_names)
region_defs = [
    (0, 'M1', [0, 1], ['RM1', 'LM1']),
    (1, 'S1', [2, 3], ['RS1', 'LS1']),
]

# Collect % and absolute deviation from experimental for each dataset/region/band
pct_deviation_records = []

for dataset_name, color, alpha, sim_label in row_defs:
    fig, axes = plt.subplots(1, 3, figsize=(12, 4),
                             gridspec_kw={'width_ratios': [1, 1, 0.4]})
    rng = np.random.default_rng(42)

    region_data = {}
    region_targets = {}

    for col_idx, region_type, rest_indices, cosim_names in region_defs:
        ax = axes[col_idx]

        if dataset_name == 'REST FIT':
            data = collect_rest_data(rest_indices, normalize=True)
        else:
            data = collect_cosim_data(cosim_names, normalize=True)

        for i, line in enumerate(data):
            ax.semilogy(freqs, line, color=color, linewidth=1, alpha=alpha,
                        label=sim_label if i == 0 else None)

        target = get_experimental_target(rest_indices, normalize=True)
        ax.semilogy(freqs, target, color=color_exp, linewidth=2, label='Exp')

        style_psd_axis(ax, col_idx, region_type, dataset_name,
                       ylabel='Normalized PSD')
        ax.set_title(region_type)

        region_data[region_type] = data
        region_targets[region_type] = target

    # ── Band-average scatter (θ and γ) ───────────────────────────────────
    ax = axes[2]

    x_pos  = [0, 0.5, 1.5, 2.0]
    labels = ['M1', 'S1', 'M1', 'S1']

    sim_band_vals = [
        np.mean(region_data['M1'][:, theta_mask], axis=1),
        np.mean(region_data['S1'][:, theta_mask], axis=1),
        np.mean(region_data['M1'][:, gamma_mask], axis=1),
        np.mean(region_data['S1'][:, gamma_mask], axis=1),
    ]
    exp_band_vals = [
        np.mean(region_targets['M1'][theta_mask]),
        np.mean(region_targets['S1'][theta_mask]),
        np.mean(region_targets['M1'][gamma_mask]),
        np.mean(region_targets['S1'][gamma_mask]),
    ]

    # % and absolute deviation from experimental
    band_region_labels = [('theta', 'M1'), ('theta', 'S1'), ('gamma', 'M1'), ('gamma', 'S1')]
    for (band_name, reg_name), sv, ev in zip(band_region_labels, sim_band_vals, exp_band_vals):
        pct_dev = (sv - ev) / ev * 100
        abs_dev = sv - ev
        mae = np.mean(np.abs(abs_dev))
        mape = np.mean(np.abs(sv - ev) / np.abs(ev)) * 100
        smape = np.mean(2 * np.abs(sv - ev) / (np.abs(sv) + np.abs(ev))) * 100
        cohens_d = (np.mean(sv) - ev) / np.std(sv) if np.std(sv) > 0 else np.nan
        pct_deviation_records.append({
            'dataset': sim_label,
            'region': reg_name,
            'band': band_name,
            'pct_dev_mean': np.mean(pct_dev),
            'pct_dev_std': np.std(pct_dev),
            'pct_dev_all': pct_dev,
            'abs_dev_mean': np.mean(abs_dev),
            'abs_dev_std': np.std(abs_dev),
            'abs_dev_all': abs_dev,
            'mae': mae,
            'mape': mape,
            'smape': smape,
            'cohens_d': cohens_d,
            'exp_val': ev,
        })

    # Combined M1+S1 records per band
    for band_name, band_mask in [('theta', theta_mask), ('gamma', gamma_mask)]:
        sv_m1 = np.mean(region_data['M1'][:, band_mask], axis=1)
        sv_s1 = np.mean(region_data['S1'][:, band_mask], axis=1)
        ev_m1 = np.mean(region_targets['M1'][band_mask])
        ev_s1 = np.mean(region_targets['S1'][band_mask])
        # Pool M1 and S1 deviations
        sv_all = np.concatenate([sv_m1, sv_s1])
        ev_all = np.concatenate([np.full_like(sv_m1, ev_m1), np.full_like(sv_s1, ev_s1)])
        pct_dev = (sv_all - ev_all) / ev_all * 100
        abs_dev = sv_all - ev_all
        mae = np.mean(np.abs(abs_dev))
        mape = np.mean(np.abs(sv_all - ev_all) / np.abs(ev_all)) * 100
        smape = np.mean(2 * np.abs(sv_all - ev_all) / (np.abs(sv_all) + np.abs(ev_all))) * 100
        cohens_d = (np.mean(sv_all - ev_all)) / np.std(sv_all - ev_all) if np.std(sv_all - ev_all) > 0 else np.nan
        pct_deviation_records.append({
            'dataset': sim_label,
            'region': 'M1+S1',
            'band': band_name,
            'pct_dev_mean': np.mean(pct_dev),
            'pct_dev_std': np.std(pct_dev),
            'pct_dev_all': pct_dev,
            'abs_dev_mean': np.mean(abs_dev),
            'abs_dev_std': np.std(abs_dev),
            'abs_dev_all': abs_dev,
            'mae': mae,
            'mape': mape,
            'smape': smape,
            'cohens_d': cohens_d,
            'exp_val': np.mean([ev_m1, ev_s1]),
        })

    for xp, sv, ev in zip(x_pos, sim_band_vals, exp_band_vals):
        jitter = rng.uniform(-0.1, 0.1, len(sv))
        ax.scatter(xp + jitter, sv, color=color, s=14, alpha=0.7, zorder=3)
        ax.hlines(ev, xp - 0.2, xp + 0.2,
                  color=color_exp, linestyle='--', linewidth=2, zorder=4)

    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels, fontsize=11)
    for center, band_label in [(0.25, 'θ'), (1.75, 'γ')]:
        ax.text(center, -0.12, band_label,
                transform=ax.get_xaxis_transform(),
                ha='center', va='top', fontsize=14, fontweight='bold')

    ax.set_title('Band averages')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.3, axis='y')

    # Legend
    legend_elements = [
        Line2D([0], [0], color=color, linewidth=1.5, label=sim_label),
        Line2D([0], [0], color=color_exp, linewidth=2, label='Exp'),
    ]
    axes[1].legend(handles=legend_elements, loc='upper right', fontsize=10)

    fig.suptitle(f'{sim_label} — Normalized PSD (G = {target_G})',
                 fontsize=14, fontweight='bold')
    fig.tight_layout()
    save_figure_multi_format(fig, f"M1S1_PSD_normalized_1x3_{sim_label.replace(' ', '_')}")
    plt.show()

print(f"\nPlots generated for G = {target_G}, Condition = {target_condition}")

# ── Print % deviation from experimental band averages ────────────────────
print("\n" + "=" * 80)
print(f"  % Deviation from experimental band averages (G = {target_G})")
print(f"  % Dev = (sim_band_avg - exp_band_avg) / exp_band_avg × 100")
print("=" * 80)
for band_name in ['theta', 'gamma']:
    band_label = 'θ (6-12 Hz)' if band_name == 'theta' else 'γ (25-47 Hz)'
    print(f"\n  Band: {band_label}")
    print(f"  {'Dataset':<12} {'Region':<6} {'Mean % dev':>12} {'Std % dev':>12} {'Min':>10} {'Max':>10}")
    print("  " + "-" * 64)
    for rec in pct_deviation_records:
        if rec['band'] == band_name:
            print(f"  {rec['dataset']:<12} {rec['region']:<6} "
                  f"{rec['pct_dev_mean']:>+11.2f}% {rec['pct_dev_std']:>11.2f}% "
                  f"{np.min(rec['pct_dev_all']):>+9.2f}% {np.max(rec['pct_dev_all']):>+9.2f}%")

# ── Print absolute deviation from experimental band averages ─────────────
print("\n" + "=" * 80)
print(f"  Absolute deviation from experimental band averages (G = {target_G})")
print(f"  Abs Dev = sim_band_avg - exp_band_avg")
print("=" * 80)
for band_name in ['theta', 'gamma']:
    band_label = 'θ (6-12 Hz)' if band_name == 'theta' else 'γ (25-47 Hz)'
    print(f"\n  Band: {band_label}")
    print(f"  {'Dataset':<12} {'Region':<6} {'Exp val':>12} {'Mean dev':>12} {'Std dev':>12} {'Min':>12} {'Max':>12}")
    print("  " + "-" * 78)
    for rec in pct_deviation_records:
        if rec['band'] == band_name:
            print(f"  {rec['dataset']:<12} {rec['region']:<6} "
                  f"{rec['exp_val']:>12.6f} "
                  f"{rec['abs_dev_mean']:>+11.6f} {rec['abs_dev_std']:>12.6f} "
                  f"{np.min(rec['abs_dev_all']):>+11.6f} {np.max(rec['abs_dev_all']):>+11.6f}")

# ── Print MAE, MAPE, sMAPE, Cohen's d summary ────────────────────────────
print("\n" + "=" * 80)
print(f"  Error metrics vs experimental band averages (G = {target_G})")
print(f"  MAE      = mean(|sim - exp|)")
print(f"  MAPE     = mean(|sim - exp| / |exp|) × 100")
print(f"  sMAPE    = mean(2·|sim - exp| / (|sim| + |exp|)) × 100")
print(f"  Cohen's d = (mean(sim) - exp) / std(sim)")
print("=" * 80)
for band_name in ['theta', 'gamma']:
    band_label = 'θ (6-12 Hz)' if band_name == 'theta' else 'γ (25-47 Hz)'
    print(f"\n  Band: {band_label}")
    header = f"  {'Dataset':<12} {'Region':<6} {'Exp val':>12} {'MAE':>12} {'MAPE (%)':>10} {'sMAPE (%)':>11} {'Cohen d':>11}"
    print(header)
    print("  " + "-" * 71)
    for rec in pct_deviation_records:
        if rec['band'] == band_name:
            print(f"  {rec['dataset']:<12} {rec['region']:<6} "
                  f"{rec['exp_val']:>12.6f} {rec['mae']:>12.6f} "
                  f"{rec['mape']:>9.2f}% {rec['smape']:>10.2f}% "
                  f"{rec['cohens_d']:>+10.3f}")

In [ ]:
# Mean ± STD – NORMALIZED
fig, axes = plt.subplots(1, 4, figsize=(10, 5))

for plot_idx, (region_type, dataset_name, rest_indices, cosim_names) in enumerate(plot_configs):
    ax = axes[plot_idx]

    if dataset_name == 'REST FIT':
        all_data = collect_rest_data(rest_indices, normalize=True)
        color = color_rest
        sim_label = 'TVMB-only'
    else:
        all_data = collect_cosim_data(cosim_names, normalize=True)
        color = color_cosim
        sim_label = 'Co-sim'

    mean_data = np.mean(all_data, axis=0)
    std_data = np.std(all_data, axis=0)
    ax.semilogy(freqs, mean_data, color=color, linewidth=2, label=sim_label)
    ax.fill_between(freqs, mean_data - std_data, mean_data + std_data,
                    color=color, alpha=0.3)

    target = get_experimental_target(rest_indices, normalize=True)
    ax.semilogy(freqs, target, color=color_exp, linewidth=2, label='Exp')

    style_psd_axis(ax, plot_idx, region_type, dataset_name,
                   ylabel='Normalized PSD',
                   show_legend=(plot_idx == 3))

plt.tight_layout()
save_figure_multi_format(fig, "M1S1_PSD_normalized_mean_std")
plt.show()
print(f"\nMean ± STD plot (NORMALIZED) generated for G = {target_G}, Condition = {target_condition}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CEREBON vs CEREBOFF — Fraction of PSD in each band (θ 6-12 Hz, γ 25-47 Hz)
# ══════════════════════════════════════════════════════════════════════════════
from scipy import stats

# ── Extract COSIM data for both conditions at target G ───────────────────────
G_idx = data_cosim['G_values'].index(target_G)

cond_on_idx  = data_cosim['conditions'].index('CEREBON')
cond_off_idx = data_cosim['conditions'].index('CEREBOFF')

cosim_on  = data_cosim['data_array'][G_idx, cond_on_idx,  :, :, :]  # (10, 20, 96)
cosim_off = data_cosim['data_array'][G_idx, cond_off_idx, :, :, :]  # (10, 20, 96)

freqs_cosim_all = data_cosim['freqs']
freq_mask = freqs_cosim_all <= 47

# Frequency-band masks on the cropped frequency vector
freqs_cropped = freqs_cosim_all[freq_mask]
theta_band = (freqs_cropped >= 6) & (freqs_cropped <= 12)
gamma_band = (freqs_cropped >= 25) & (freqs_cropped <= 47)

region_idx_map = data_cosim['region_idx_map']
region_groups = {
    'M1': ['RM1', 'LM1'],
    'S1': ['RS1', 'LS1'],
}

# ── Compute fraction of PSD in each band per repetition ─────────────────────
# Fraction = sum(PSD in band) / sum(PSD over all frequencies in 5–47 Hz)
results = {}  # results[(region, band, condition)] = array of shape (n_reps*n_hemi,)

for region_label, region_names in region_groups.items():
    for band_name, band_mask in [('theta', theta_band), ('gamma', gamma_band)]:
        for cond_name, cond_data in [('CEREBON', cosim_on), ('CEREBOFF', cosim_off)]:
            vals = []
            for rep in range(cond_data.shape[0]):
                for rname in region_names:
                    ridx = region_idx_map[rname]
                    psd = cond_data[rep, ridx, freq_mask]
                    fraction = np.sum(psd[band_mask]) / np.sum(psd)
                    vals.append(fraction)
            results[(region_label, band_name, cond_name)] = np.array(vals)

# ── Statistical comparison (independent t-test, 2-sided) ────────────────────
print("=" * 70)
print(f"  CEREBON vs CEREBOFF — Band power fraction comparison (G = {target_G})")
print(f"  Fraction = sum(PSD in band) / sum(PSD over 5-47 Hz)")
print("=" * 70)

for region_label in ['M1', 'S1']:
    print(f"\n  Region: {region_label}")
    print(f"  {'Band':<8} {'CEREBON (mean +/- std)':>24} {'CEREBOFF (mean +/- std)':>24} "
          f"{'t-stat':>8} {'p-value':>10} {'sig':>5}")
    print("  " + "-" * 80)
    for band_name in ['theta', 'gamma']:
        on_vals  = results[(region_label, band_name, 'CEREBON')]
        off_vals = results[(region_label, band_name, 'CEREBOFF')]
        t_stat, p_val = stats.ttest_ind(on_vals, off_vals)
        sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'n.s.'
        print(f"  {band_name:<8} {np.mean(on_vals):>10.4f} +/- {np.std(on_vals):<10.4f}"
              f" {np.mean(off_vals):>10.4f} +/- {np.std(off_vals):<10.4f}"
              f" {t_stat:>8.3f} {p_val:>10.2e} {sig:>5}")

# ── Visualization ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

color_on  = viridis(0.75)   # green - same as existing COSIM color
color_off = viridis(0.45)   # teal

rng = np.random.default_rng(42)

for ax_idx, region_label in enumerate(['M1', 'S1']):
    ax = axes[ax_idx]

    x_positions = [0, 1, 3, 4]
    tick_labels = ['ON', 'OFF', 'ON', 'OFF']

    data_sets = [
        results[(region_label, 'theta', 'CEREBON')],
        results[(region_label, 'theta', 'CEREBOFF')],
        results[(region_label, 'gamma', 'CEREBON')],
        results[(region_label, 'gamma', 'CEREBOFF')],
    ]
    colors = [color_on, color_off, color_on, color_off]

    for xp, dset, c in zip(x_positions, data_sets, colors):
        jitter = rng.uniform(-0.15, 0.15, len(dset))
        ax.scatter(xp + jitter, dset, color=c, s=20, alpha=0.7, zorder=3)
        ax.hlines(np.mean(dset), xp - 0.25, xp + 0.25,
                  color=c, linewidth=2, zorder=4)

    # Significance brackets
    for band_name, x_left, x_right in [('theta', 0, 1), ('gamma', 3, 4)]:
        on_vals  = results[(region_label, band_name, 'CEREBON')]
        off_vals = results[(region_label, band_name, 'CEREBOFF')]
        _, p_val = stats.ttest_ind(on_vals, off_vals)
        sig_label = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'n.s.'
        y_max = max(np.max(on_vals), np.max(off_vals))
        bracket_y = y_max * 1.08
        ax.plot([x_left, x_left, x_right, x_right],
                [bracket_y * 0.98, bracket_y, bracket_y, bracket_y * 0.98],
                color='black', linewidth=1)
        ax.text((x_left + x_right) / 2, bracket_y * 1.02, sig_label,
                ha='center', va='bottom', fontsize=11)

    ax.set_xticks(x_positions)
    ax.set_xticklabels(tick_labels, fontsize=11)
    # Group labels
    for center, band_label in [(0.5, 'theta (6-12 Hz)'), (3.5, 'gamma (25-47 Hz)')]:
        ax.text(center, -0.10, band_label,
                transform=ax.get_xaxis_transform(),
                ha='center', va='top', fontsize=12, fontweight='bold')

    ax.set_title(region_label, fontsize=14)
    if ax_idx == 0:
        ax.set_ylabel('Fraction of total PSD', fontsize=13)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.3, axis='y')

# Legend
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=color_on,
           markersize=8, label='CEREBON'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=color_off,
           markersize=8, label='CEREBOFF'),
]
axes[1].legend(handles=legend_elements, loc='upper right', fontsize=11)

fig.suptitle(f'CEREBON vs CEREBOFF - Fraction of PSD per band (G = {target_G})',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_figure_multi_format(fig, "CEREBON_vs_CEREBOFF_band_fraction")
plt.show()

# ── Also plot full PSD lines for both conditions (mean +/- std) ──────────────
fig2, axes2 = plt.subplots(1, 2, figsize=(10, 4.5))

for ax_idx, (region_label, region_names) in enumerate(region_groups.items()):
    ax = axes2[ax_idx]

    for cond_name, cond_data, c in [('CEREBON', cosim_on, color_on),
                                     ('CEREBOFF', cosim_off, color_off)]:
        all_psd = []
        for rep in range(cond_data.shape[0]):
            for rname in region_names:
                ridx = region_idx_map[rname]
                psd = cond_data[rep, ridx, freq_mask]
                psd_norm = psd / np.sum(psd)
                all_psd.append(psd_norm)
        all_psd = np.array(all_psd)
        mean_psd = np.mean(all_psd, axis=0)
        std_psd  = np.std(all_psd, axis=0)

        ax.semilogy(freqs_cropped, mean_psd, color=c, linewidth=2, label=cond_name)
        ax.fill_between(freqs_cropped, mean_psd - std_psd, mean_psd + std_psd,
                        color=c, alpha=0.25)

    style_psd_axis(ax, ax_idx, region_label, 'COSIM',
                   ylabel='Normalized PSD', xlim=(5, 47))
    ax.set_title(region_label, fontsize=14)

legend_elements2 = [
    Line2D([0], [0], color=color_on, linewidth=2, label='CEREBON'),
    Line2D([0], [0], color=color_off, linewidth=2, label='CEREBOFF'),
]
axes2[1].legend(handles=legend_elements2, loc='upper right', fontsize=11)

fig2.suptitle(f'CEREBON vs CEREBOFF - Normalized PSD (G = {target_G})',
              fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_figure_multi_format(fig2, "CEREBON_vs_CEREBOFF_PSD_lines")
plt.show()